In [19]:
import numpy as np
from LanzaModels import TVL1_1D
from ADMMsRustici import MyBackTrackingSolver
from signalClass import *
import time

In [20]:
np.random.seed(24102000)
n = 128

construct blur matrix

In [21]:
#blur matrix construction

a = 0.25
b = 0.5
c = 0.25

diagB = b * np.ones(shape=(n,))
offDiagA = a * np.ones(shape=(n-1,))
offDiagC = c * np.ones(shape=(n-1,))
A = np.diag(diagB, 0) + np.diag(offDiagC, 1) + np.diag(offDiagA, -1)

#apply anti-reflexive BCs

A[0][0] = 2 * a + b
A[0][1] = c - a
A[n-1][n-2] = a - c
A[n-1][n-1] = b + 2 * c

#end blur matrix construction

construct signal

In [22]:
#begin signal construction

PwSignal = signal(n)
RndSignal = signal(n)
sigma = 0.01

PwSignal.generate_cartoon_sign(2, 15)
RndSignal.generate_GG_realization(0, sigma, 2)

xTrue = PwSignal.get_image()
xCorrupted = (A @ xTrue) + RndSignal.get_image()

#end signal construction

Define the TVL2 model

In [23]:
mu = 0.5
VarModel = TVL1_1D.TVL1_1DClass(A, xCorrupted, mu)

Lfid = mu * np.max(np.linalg.svdvals(A)) * np.sqrt(n)
Lreg = np.sqrt(n)
Lphi = np.sqrt(Lfid**2 + Lreg**2)

Now, we need to initialize and define the solver

In [24]:
#begin solver construction
np.random.seed(24102002)

xk = np.random.randn(n,)
yk = np.random.randn(n,)
betak = 1
lk = np.zeros(n)

x0 = xk.copy()
y0 = yk.copy()

MySolver = MyBackTrackingSolver.MyBacktrackingSolverClass(VarModel, xk, yk, lk, betak, Lphi)

#end solver construction

In [25]:
iters = 11

XsolutionHistory = np.zeros(shape=(iters, n))
YsolutionHistory = np.zeros(shape=(iters, n))

lambdaHistory = np.zeros(shape=(iters, n))

betaHistory = np.zeros(shape=(iters,))

PrimalResidueHistory = np.zeros(shape=(iters,))
DualResidueHistory = np.zeros(shape=(iters,))
ImgHistory = np.zeros(shape=(iters,))

CpuTimes = np.zeros(shape=(iters,))

In [26]:
timer = 0

for iter in range(0, iters):

    print(f"{iter + 1} / {iters}")

    sTime = time.perf_counter_ns()

    xk_1, yk_1, lk_1, betak_1, dualResk = MySolver.CallMyIterationStep(xk, yk, lk, betak)

    eTime = time.perf_counter_ns()

    timer += ( (eTime - sTime) / 1e9 )

    
    primalResidue = np.linalg.norm(VarModel.P @ xk_1 + VarModel.Q @ yk_1 - VarModel.c)
    print(f"primalRes: {primalResidue}")

    XsolutionHistory[iter, :] = xk_1
    YsolutionHistory[iter, :] = yk_1
    lambdaHistory[iter, :] = lk_1
    betaHistory[iter] = betak_1

    PrimalResidueHistory[iter] = primalResidue
    DualResidueHistory[iter] = dualResk
    ImgHistory[iter] = VarModel(xk_1, yk_1)
    CpuTimes[iter] = timer

    xk = xk_1
    yk = yk_1
    lk = lk_1
    betak = betak_1

    if (max(primalResidue, dualResk) <= 1e-9):
        print(iter)
        break


1 / 11
iter Mnon: 5
delta L: 0.7381764665906769
iter Backtrack: 82
DualRes: 3.780656644572304e-08
Diamk: 13.177271590567738
primalRes: 0.9922569292466372
2 / 11
iter Mnon: 21
delta L: 0.11993305783972419
iter Backtrack: 24
DualRes: 3.900444601252957e-08
Diamk: 7.833522734972033
primalRes: 0.26776465081984396
3 / 11
iter Mnon: 10
delta L: 0.0228899096044346
iter Backtrack: 67
DualRes: 9.529528174594731e-08
Diamk: 4.898890206086368
primalRes: 0.10451141444753224
4 / 11
iter Mnon: 27
delta L: 0.002441353759119025
iter Backtrack: 96
DualRes: 1.5656861086440566e-07
Diamk: 2.9933722369977
primalRes: 0.051569685989183905
5 / 11
iter Mnon: 32
delta L: 0.0007449860401527175
iter Backtrack: 1310
DualRes: 2.597749617782053e-07
Diamk: 1.902887225191197
primalRes: 0.02775930353829677
6 / 11
iter Mnon: 57
delta L: 0.00023905918261668546
iter Backtrack: 194
DualRes: 3.847450592706669e-07
Diamk: 1.223941645860133
primalRes: 0.013809190975960646
7 / 11
iter Mnon: 58
delta L: 5.791080602080001e-05
iter 

In [27]:
np.savez_compressed(
    "./MyADMMTVL1-Gauss.npz",
    Xs = XsolutionHistory,
    Ys = YsolutionHistory,
    Ls = lambdaHistory,
    Betas = betaHistory,

    PrimalRes = PrimalResidueHistory,
    DualRes = DualResidueHistory,
	IMGs = ImgHistory,
    CpuTimes = CpuTimes,

    xTrue = xTrue,
    xCorrupted = xCorrupted,
	x0 = x0,
	y0 = y0
)